# WFGY MVP Experiment: Q109 — Migration Dynamics

**Problem**: TU Q109 — Can a scalar tension observable track destabilisation in large-scale migration flows?

## Data Source
- **Dataset**: World Bank Open Data — Net Migration (`SM.POP.NETM`)
- **API**: `https://api.worldbank.org/v2/country/all/indicator/SM.POP.NETM`
- **License**: CC BY 4.0 (open access, free to use with attribution)
- **Coverage**: ~190 countries, most recent available year

## WFGY Framework
- **G (Anchor)**: Global median net migration — baseline of a 'neutral' state
- **I (Current)**: Each country's net migration flow
- **ΔS**: Normalised deviation from the anchor — measures structural stress

In [ ]:
import urllib.request
import json
import math

# ----- 1. Load Data from World Bank Open Data API -----
# SM.POP.NETM = Net migration (net number of migrants)
WB_URL = ('https://api.worldbank.org/v2/country/all/indicator/SM.POP.NETM'
          '?format=json&mrv=1&per_page=50')

print(f'Fetching from World Bank API...')
req = urllib.request.Request(WB_URL, headers={'User-Agent': 'WFGY-MVP/1.0'})
with urllib.request.urlopen(req, timeout=15) as resp:
    data = json.loads(resp.read().decode('utf-8'))

# Filter out aggregate/regional groups (they have numeric-style ISO codes)
EXCLUDE_KEYWORDS = ['income', 'IBRD', 'IDA', 'OECD', 'World', 'dividend', 'Fragile', 'Heavily', 'blend', 'small state', 'Sub-Saharan', 'Latin', 'Europe', 'Asia', 'Middle East']

records = []
for r in data[1]:
    if r['value'] is None:
        continue
    name = r['country']['value']
    if any(kw.lower() in name.lower() for kw in EXCLUDE_KEYWORDS):
        continue
    records.append((name, r['value'], r['date']))

print(f'Country records loaded: {len(records)}')
print(f'Reference year: {records[0][2]}')

In [ ]:
import statistics

# ----- 2. Compute WFGY ΔS Tension -----
flows = [v for _, v, _ in records]
G_anchor = statistics.median(flows)  # Anchor: global median
print(f'Anchor G (global median net migration): {G_anchor:,.0f}\n')

def delta_s(flow, anchor):
    """WFGY ΔS proxy: normalised deviation from the equilibrium anchor."""
    magnitude = max(abs(anchor), 1)
    deviation = abs(flow - anchor) / magnitude
    return min(1.0, deviation / (deviation + 1))

def zone_label(ds):
    if ds >= 0.85:   return 'DANGER'
    elif ds >= 0.60: return 'RISK'
    elif ds >= 0.40: return 'TRANSITIONAL'
    else:            return 'SAFE'

results = [(name, flow, delta_s(flow, G_anchor), zone_label(delta_s(flow, G_anchor))) for name, flow, _ in records]
results.sort(key=lambda x: x[2], reverse=True)

print(f'{'Country':<35} {'Net Migration':>16}  {'ΔS':>6}  {'Zone'}')
print('-' * 72)
for name, flow, ds, zone in results[:15]:
    print(f'{name:<35} {flow:>16,.0f}  {ds:>6.3f}  {zone}')

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# ----- 3. Visualise ΔS Tension Map -----
top_n = results[:15]
countries = [r[0] for r in top_n]
ds_values  = [r[2] for r in top_n]
colors = ['darkred' if ds >= 0.85 else 'orange' if ds >= 0.60 else 'steelblue' for ds in ds_values]

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.barh(countries[::-1], ds_values[::-1], color=colors[::-1])

ax.axvline(x=0.85, color='darkred',  linestyle='--', linewidth=1.2, label='Danger Zone (ΔS ≥ 0.85)')
ax.axvline(x=0.60, color='orange',   linestyle='--', linewidth=1.2, label='Risk Zone (ΔS ≥ 0.60)')
ax.set_xlabel('ΔS Tension Index')
ax.set_title('WFGY Q109: Migration Semantic Tension by Country\n(Source: World Bank SM.POP.NETM)', fontsize=13)
ax.set_xlim(0, 1.05)
ax.legend()
plt.tight_layout()
plt.show()

# Summary
danger_countries = [r[0] for r in results if r[3] == 'DANGER']
risk_countries   = [r[0] for r in results if r[3] == 'RISK']
print(f'\nCountries in DANGER zone (ΔS ≥ 0.85): {len(danger_countries)}')
print(f'Countries in RISK zone   (ΔS ≥ 0.60): {len(risk_countries)}')

## 4. Interpretation

| Zone | ΔS Range | Meaning |
|---|---|---|
| SAFE | < 0.40 | Migration near global equilibrium |
| TRANSITIONAL | 0.40–0.60 | Moderate drift, worth tracking |
| RISK | 0.60–0.85 | Systemic stress — policy action warranted |
| DANGER | ≥ 0.85 | Structural collapse risk — BBCR trigger |

Countries with extreme **positive** net migration (major destinations) and extreme **negative** net migration (conflict/crisis origins) both generate high ΔS — the tension is symmetric, capturing both sides of the migration corridor.

This directly maps to **Q109's core thesis**: the tension between origin-push and destination-pull creates observable, measurable stress in the global migration system.